In [7]:
from pymongo import MongoClient
import pandas as pd
import json

In [8]:
# # Configuração do MongoDB
# MONGO_URI = "mongodb://core-local-4p-aura-brain-pro-cosmosdb:SWbSbRw7K8z7ODssj9q0CKW4yWy6glIbeV1Ttqom83KlDWNya5DNMQConwJGep8ngDTax7gYtO6yACDb1Plipw%3D%3D@core-local-4p-aura-brain-pro-cosmosdb.mongo.cosmos.azure.com:10255/ms-embeddings-api?ssl=true&retrywrites=false&replicaSet=globaldb&maxIdleTimeMS=120000&appName=@core-local-4p-aura-brain-pro-cosmosdb@"
# DATABASE = "ms-embeddings-api"
# COLLECTION = "knowledge-base"
# LIMIT = 5  # quantidade padrão


# # Conexão com o MongoDB
# client = MongoClient(MONGO_URI)
# db = client[DATABASE]
# collection = db[COLLECTION]

In [16]:
from pymongo import MongoClient
import pandas as pd
import json
import os
import time
from datetime import datetime

# ==================================================
# CONFIGURAÇÕES
# ==================================================

MONGO_URI = "mongodb://core-local-4p-aura-brain-pro-cosmosdb:SWbSbRw7K8z7ODssj9q0CKW4yWy6glIbeV1Ttqom83KlDWNya5DNMQConwJGep8ngDTax7gYtO6yACDb1Plipw%3D%3D@core-local-4p-aura-brain-pro-cosmosdb.mongo.cosmos.azure.com:10255/ms-embeddings-api?ssl=true&retrywrites=false&replicaSet=globaldb&maxIdleTimeMS=120000&appName=@core-local-4p-aura-brain-pro-cosmosdb@"
DATABASE = "ms-embeddings-api"
COLLECTION = "knowledge-base"
LIMIT = 5  # quantidade padrão

SUBSCRIPTION = "6495aca06e4f0f95368f289c"

LIMIT = 4000

DELAY_SECONDS = 0.1

SALVAR_CADA = 100

XLSX_FILE = "resultado_vivo_file_force.xlsx"
CHECKPOINT_FILE = "checkpoint.json"
LOG_FILE = "processamento.log"

# ==================================================
# LOG
# ==================================================


def log(msg):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    mensagem = f"[{timestamp}] {msg}"

    print(mensagem)

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(mensagem + "\n")


# ==================================================
# CHECKPOINT
# ==================================================


def carregar_checkpoint():

    if os.path.exists(CHECKPOINT_FILE):

        with open(CHECKPOINT_FILE, "r") as f:

            return set(json.load(f).get("processed_ids", []))

    return set()


def salvar_checkpoint(processed_ids):

    with open(CHECKPOINT_FILE, "w") as f:

        json.dump({"processed_ids": list(processed_ids)}, f, indent=4)


# ==================================================
# XLSX
# ==================================================


def carregar_planilha():

    if os.path.exists(XLSX_FILE):

        df = pd.read_excel(XLSX_FILE)

        return df

    return pd.DataFrame(
        columns=[
            "Id",
            "Nome do Arquivo",
            "FileUrl",
            "Vivo_File_Force",
            "Status",
            "Data_Processamento",
        ]
    )


# ==================================================
# INÍCIO
# ==================================================

log("=" * 80)
log("INICIANDO PROCESSAMENTO")
log("=" * 80)

client = MongoClient(MONGO_URI)

db = client[DATABASE]

collection = db[COLLECTION]

query = {"name": {"$regex": r"\.html"}, "subscription": SUBSCRIPTION}

total_documentos = collection.count_documents(query)

log(f"Total encontrado: " f"{total_documentos}")

# ==================================================
# CHECKPOINT
# ==================================================

processed_ids = carregar_checkpoint()

log(f"Checkpoint carregado com " f"{len(processed_ids)} registros")

# ==================================================
# XLSX
# ==================================================

df = carregar_planilha()

# IDs já existentes na planilha
ids_planilha = {}

if not df.empty:

    for _, row in df.iterrows():

        ids_planilha[str(row["Id"])] = row["Status"]

# ==================================================
# CURSOR
# ==================================================

cursor = collection.find(query, no_cursor_timeout=True).batch_size(100)

# ==================================================
# CONTADORES
# ==================================================

processados = 0
ignorados = 0
encontrados = 0
nao_encontrados = 0

# ==================================================
# LOOP PRINCIPAL
# ==================================================

for doc in cursor:

    if processados >= LIMIT:
        break

    try:

        doc_id = str(doc.get("_id"))

        # ==================================
        # Ignora apenas se estiver marcado
        # explicitamente como Processado
        # ==================================

        if doc_id in ids_planilha and ids_planilha[doc_id] == "Processado":

            ignorados += 1

            log(f"SKIP -> " f"{doc.get('name')}")

            continue

        nome_arquivo = doc.get("name", "Sem nome")

        file_url = doc.get("fileUrl", "")

        # ----------------------------------
        # BUSCA
        # ----------------------------------

        documento_texto = json.dumps(doc, default=str)

        possui_vivo_force = "vivo.file.force" in documento_texto

        if possui_vivo_force:
            encontrados += 1
        else:
            nao_encontrados += 1

        nova_linha = {
            "Id": doc_id,
            "Nome do Arquivo": nome_arquivo,
            "FileUrl": file_url,
            "Vivo_File_Force": "Sim" if possui_vivo_force else "Não",
            "Status": "Processado",
            "Data_Processamento": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

        # Remove versão anterior se existir

        if doc_id in df["Id"].astype(str).values:

            df = df[df["Id"].astype(str) != doc_id]

        # Adiciona nova linha

        df = pd.concat([df, pd.DataFrame([nova_linha])], ignore_index=True)

        processed_ids.add(doc_id)

        processados += 1

        percentual = (processados / total_documentos) * 100

        log(
            f"[{processados}/{total_documentos}] "
            f"{percentual:.2f}% "
            f"Arquivo={nome_arquivo} "
            f"Vivo_File_Force="
            f"{'Sim' if possui_vivo_force else 'Não'}"
        )

        # ==================================
        # SAVE BATCH
        # ==================================

        if processados % SALVAR_CADA == 0:

            df.to_excel(XLSX_FILE, index=False)

            salvar_checkpoint(processed_ids)

            log(f"Checkpoint salvo " f"com {processados} " f"processados.")

        # ==================================
        # DELAY
        # ==================================

        time.sleep(DELAY_SECONDS)

    except Exception as e:

        log(f"ERRO: " f"{doc.get('_id')} " f"-> {str(e)}")

        continue

# ==================================================
# FINALIZAÇÃO
# ==================================================

df.to_excel(XLSX_FILE, index=False)

salvar_checkpoint(processed_ids)

log("=" * 80)
log("PROCESSAMENTO FINALIZADO")
log("=" * 80)

log(f"Total Processados: " f"{processados}")

log(f"Encontrados VivoForce: " f"{encontrados}")

log(f"Não Encontrados: " f"{nao_encontrados}")

log(f"Ignorados: " f"{ignorados}")

log(f"Planilha gerada: " f"{XLSX_FILE}")

2026-07-15 17:40:51,842 - INFO - You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb


[2026-07-15 17:40:51] ================================================================================
[2026-07-15 17:40:51] INICIANDO PROCESSAMENTO
[2026-07-15 17:40:51] ================================================================================
[2026-07-15 17:40:52] Total encontrado: 3037
[2026-07-15 17:40:52] Checkpoint carregado com 3037 registros
[2026-07-15 17:40:53] [1/3037] 0.03% Arquivo=ka0Dy0000009vV9IAI-Guilherme.html Vivo_File_Force=Não
[2026-07-15 17:40:53] [2/3037] 0.07% Arquivo=ka0Dy0000009yZPIAY-TESTE-APENAS-COM-IMAGEM-NO-DESCRI.html Vivo_File_Force=Não
[2026-07-15 17:40:53] [3/3037] 0.10% Arquivo=ka0Hq0000019ta9IAA-TESTE-APENAS-COM-IMAGEM-NO-DESCRI.html Vivo_File_Force=Não
[2026-07-15 17:40:53] [4/3037] 0.13% Arquivo=ka0Hq0000019tpZIAQ-PR-Fixa-Legado-B2B-Bloqueador-de-Celular-(BLG-e-BLC).html Vivo_File_Force=Não
[2026-07-15 17:40:53] [5/3037] 0.16% Arquivo=ka0Hq0000019tqrIAA-teste-10-10.html Vivo_File_Force=Não
[2026-07-15 17:40:53] [6/3037] 0.20% Arquivo=ka0Hq000

In [ ]:
# from pymongo import MongoClient
# import pandas as pd
# import json
# import os
# import logging

# # ==================================================
# # CONFIGURAÇÕES
# # ==================================================

# MONGO_URI = "mongodb://core-local-4p-aura-brain-pro-cosmosdb:SWbSbRw7K8z7ODssj9q0CKW4yWy6glIbeV1Ttqom83KlDWNya5DNMQConwJGep8ngDTax7gYtO6yACDb1Plipw%3D%3D@core-local-4p-aura-brain-pro-cosmosdb.mongo.cosmos.azure.com:10255/ms-embeddings-api?ssl=true&retrywrites=false&replicaSet=globaldb&maxIdleTimeMS=120000&appName=@core-local-4p-aura-brain-pro-cosmosdb@"
# DATABASE = "ms-embeddings-api"
# COLLECTION = "knowledge-base"

# LIMIT = 20

# CHECKPOINT_FILE = "checkpoint.json"
# EXCEL_FILE = "resultado_vivo_file_force.xlsx"

# # ==================================================
# # LOGGING
# # ==================================================

# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s - %(levelname)s - %(message)s",
#     handlers=[logging.FileHandler("processamento.log"), logging.StreamHandler()],
# )

# # ==================================================
# # CHECKPOINT
# # ==================================================


# def carregar_checkpoint():
#     if os.path.exists(CHECKPOINT_FILE):
#         with open(CHECKPOINT_FILE, "r") as f:
#             return set(json.load(f).get("processed_ids", []))
#     return set()


# def salvar_checkpoint(processed_ids):
#     with open(CHECKPOINT_FILE, "w") as f:
#         json.dump({"processed_ids": list(processed_ids)}, f, indent=4)


# # ==================================================
# # EXCEL
# # ==================================================


# def carregar_resultados_existentes():

#     if os.path.exists(EXCEL_FILE):
#         return pd.read_excel(EXCEL_FILE)

#     return pd.DataFrame(columns=["Id", "Nome do Arquivo", "FileUrl", "Vivo_File_Force"])


# # ==================================================
# # MONGO
# # ==================================================

# client = MongoClient(MONGO_URI)

# db = client[DATABASE]
# collection = db[COLLECTION]

# query = {"name": {"$regex": r"\.html"}, "subscription": "6495aca06e4f0f95368f289c"}

# # ==================================================
# # PROCESSAMENTO
# # ==================================================

# processed_ids = carregar_checkpoint()

# logging.info(f"Checkpoint carregado com {len(processed_ids)} registros.")

# df_resultado = carregar_resultados_existentes()

# cursor = collection.find(query)

# processados_nesta_execucao = 0

# for doc in cursor:

#     if processados_nesta_execucao >= LIMIT:
#         break

#     doc_id = str(doc["_id"])

#     # pula já processados
#     if doc_id in processed_ids:
#         continue

#     nome_arquivo = doc.get("name", "")
#     file_url = doc.get("fileUrl", "")

#     documento_texto = json.dumps(doc, default=str)

#     possui_vivo_force = "vivo.file.force" in documento_texto

#     nova_linha = {
#         "Id": doc_id,
#         "Nome do Arquivo": nome_arquivo,
#         "FileUrl": file_url,
#         "Vivo_File_Force": ("Sim" if possui_vivo_force else "Não"),
#     }

#     df_resultado = pd.concat(
#         [df_resultado, pd.DataFrame([nova_linha])], ignore_index=True
#     )

#     processed_ids.add(doc_id)

#     salvar_checkpoint(processed_ids)

#     processados_nesta_execucao += 1

#     logging.info(
#         f"[{processados_nesta_execucao}/{LIMIT}] " f"Processado: {nome_arquivo}"
#     )

# # ==================================================
# # PERSISTE EXCEL
# # ==================================================

# df_resultado.to_excel(EXCEL_FILE, index=False)

# logging.info(f"Planilha atualizada: {EXCEL_FILE}")

# logging.info(f"Total processados nesta execução: " f"{processados_nesta_execucao}")

2026-07-15 16:30:02,706 - INFO - You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
2026-07-15 16:30:02,755 - INFO - Checkpoint carregado com 5 registros.
2026-07-15 16:30:04,082 - INFO - [1/20] Processado: ka0Hq0000019tqqIAA-artigo-novo-teste-10-10.html
2026-07-15 16:30:04,087 - INFO - [2/20] Processado: ka0Hq0000019tr5IAA-PR-Fixa-Legado-B2B-Funcionamento-do-Aparelho-Chamadas-TEFWT.html
2026-07-15 16:30:04,091 - INFO - [3/20] Processado: ka0Hq0000019tr0IAA-PR-Fixa-Legado-B2B-Chip-Only-FWT-dentro-de-Sao-Paulo.html
2026-07-15 16:30:04,095 - INFO - [4/20] Processado: ka0Hq0000019trtIAA-PR-Convergente-B2B-(Movel-Fixa-Avancada)-Geracao-Distribuida-Geracao-Compartilhada-da-GUD-Energia.html
2026-07-15 16:30:04,100 - INFO - [5/20] Processado: ka0Hq0000019trjIAA-PR-Fixa-Legado-B2B-Cancelamento-e-Melhor-Oferta.html
2026-07-15 16:30:04,104 - INFO - [6/20] Processado: ka0Hq